In [1]:
import io
import zipfile
from pathlib import PurePosixPath

import numpy as np
import pandas as pd
import sklearn
import tensorflow as tf
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from tensorflow import keras
from tensorflow.keras import layers


In [2]:
train_zip_path = "data/train_images_small.zip"
test_zip_path = "data/test_images_small.zip"
csv_path = "data/train_images_metadata.csv"

df = pd.read_csv(csv_path)

In [3]:
# 1. Scan train_images_small.zip and build a lookup dictionary
train_zip_map = {}

with zipfile.ZipFile(train_zip_path, "r") as z:
    for internal_path in z.namelist():
        if internal_path.lower().endswith(
            (".jpg", ".jpeg")
        ):
            filename = PurePosixPath(internal_path).name
            train_zip_map[filename] = internal_path

# 3. Filter out rows from dataframe that only exist in the metadata but not in train_images_small.zip
print(f"Total rows in CSV: {len(df)}")
df = df[df["image_path"].isin(train_zip_map)].reset_index(drop=True)
print(f"Matched images found in zip: {len(df)}")


Total rows in CSV: 66454
Matched images found in zip: 66060


In [4]:
df[:5]

,Unnamed: 0,observation_id,endemic,binomial_name,code,image_path,class_id
0,0,117940,False,Bitis caudalis,ZA,168482.JPG,157
1,1,37280129,False,Amphiesma stolatum,TH,59067968.jpg,66
2,2,27445689,False,Lachesis muta,BR,42704088.jpg,857
3,3,68796818,True,Vipera berus,NL,111487306.jpeg,1736
4,4,68796816,False,Coronella austriaca,NL,111486989.jpeg,387


In [5]:
snake_classes = df["class_id"].nunique()
snake_classes

296

In [6]:
IMG_SIZE = 224  # 224x224 or 128x128 trains fast for a custom CNN

def create_cnn(num_classes, img_size = 224):
    cnn = keras.Sequential(
        [
            keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3)),
            # Normalize pixel values from [0, 255] to [0.0, 1.0]
            layers.Rescaling(1.0 / 255),
            # Light data augmentation so it doesn't memorize training photos
            layers.RandomFlip("horizontal"),
            layers.RandomRotation(0.05),
            # Block 1: Detects simple edges and color gradients
            layers.Conv2D(32, kernel_size=(3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(2, 2)),
            # Block 2: Detects scale textures and stripes/bands
            layers.Conv2D(64, kernel_size=(3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(2, 2)),
            # Block 3: Detects complex patterns (head shapes, dorsal diamonds)
            layers.Conv2D(128, kernel_size=(3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.MaxPooling2D(pool_size=(2, 2)),
            # Block 4: High-level species features
            layers.Conv2D(256, kernel_size=(3, 3), padding="same", activation="relu"),
            layers.BatchNormalization(),
            layers.GlobalAveragePooling2D(),  # Better & smaller than Flatten()!
            # Classifier Head
            layers.Dropout(0.4),
            layers.Dense(128, activation="relu"),
            layers.Dense(num_classes, activation="softmax"),
        ]
    )
    return cnn


In [7]:
cnn = create_cnn(num_classes=snake_classes)

cnn.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

cnn.summary()


Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 rescaling (Rescaling)       (None, 224, 224, 3)       0         
                                                                 
 random_flip (RandomFlip)    (None, 224, 224, 3)       0         
                                                                 
 random_rotation (RandomRota  (None, 224, 224, 3)      0         
 tion)                                                           
                                                                 
 conv2d (Conv2D)             (None, 224, 224, 32)      896       
                                                                 
 batch_normalization (BatchN  (None, 224, 224, 32)     128       
 ormalization)                                                   
                                                                 
 max_pooling2d (MaxPooling2D  (None, 112, 112, 32)     0

___

In [8]:
# 1. Re-map class_id to contiguous integers (0 to 295) for TensorFlow
df["model_label"], unique_classes = pd.factorize(df["class_id"])

# 2. Split into 80% Train / 20% Val, grouping by observation_id
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, val_idx = next(gss.split(df, groups=df["observation_id"]))

train_df = df.iloc[train_idx].reset_index(drop=True)
val_df = df.iloc[val_idx].reset_index(drop=True)

print(f"Training images: {len(train_df)}")
print(f"Validation images: {len(val_df)}")

Training images: 52701
Validation images: 13359


In [ ]:
class ZipSnakeDataset(keras.utils.Sequence):
    def __init__(
        self,
        dataframe,
        zip_path,
        zip_map,
        batch_size=32,
        img_size=(224, 224),
        shuffle=True,
        **kwargs,
    ):
        super().__init__(**kwargs)
        self.df = dataframe
        self.zip_path = zip_path
        self.zip_map = zip_map
        self.batch_size = batch_size
        self.img_size = img_size
        self.shuffle = shuffle
        self.indices = np.arange(len(self.df))
        self.zip_handle = None
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __len__(self):
        return int(np.ceil(len(self.df) / self.batch_size))

    def __getitem__(self, idx):
        # Open zip file safely per worker
        if self.zip_handle is None:
            self.zip_handle = zipfile.ZipFile(self.zip_path, "r")

        # Get batch row indices
        batch_indices = self.indices[
            idx * self.batch_size : (idx + 1) * self.batch_size
        ]
        batch_rows = self.df.iloc[batch_indices]

        images, labels = [], []
        for _, row in batch_rows.iterrows():
            # Lookup the exact internal zip path using your train_zip_map
            internal_path = self.zip_map[row["image_path"]]
            raw_bytes = self.zip_handle.read(internal_path)

            # Decode and resize to 224x224
            img = Image.open(io.BytesIO(raw_bytes)).convert("RGB").resize(self.img_size)
            images.append(np.array(img, dtype=np.float32))
            labels.append(row["model_label"])

        return np.array(images), np.array(labels, dtype=np.int32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)


# Initialize the loaders (using the train_zip_map you already built)
train_loader = ZipSnakeDataset(
    train_df, train_zip_path, train_zip_map, batch_size=64, shuffle=True
)
val_loader = ZipSnakeDataset(
    val_df, train_zip_path, train_zip_map, batch_size=64, shuffle=False
)


In [10]:
history = cnn.fit(train_loader, validation_data=val_loader, epochs=30)

Epoch 1/15
824/824 [==============================] - 363s 427ms/step - loss: 5.3531 - accuracy: 0.0353 - val_loss: 5.3778 - val_accuracy: 0.0353
Epoch 2/15
824/824 [==============================] - 335s 407ms/step - loss: 5.2062 - accuracy: 0.0420 - val_loss: 5.2614 - val_accuracy: 0.0418
Epoch 3/15
824/824 [==============================] - 335s 407ms/step - loss: 5.1185 - accuracy: 0.0469 - val_loss: 5.3159 - val_accuracy: 0.0378
Epoch 4/15
801/824 [============================>.] - ETA: 8s - loss: 5.0234 - accuracy: 0.0519

KeyboardInterrupt: 